In [3]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
load_dotenv()

from utility.env_util import get_api_key

find_api = "OPENAI_API_KEY"
api_key = get_api_key(find_api)

In [4]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,  # 말이 길어지는 걸 방지
    max_tokens=100,   # 응답 최대 길이 제한
)

In [5]:
# 세션별 대화 기록을 저장할 딕셔너리
session_storage = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 session_storage에 없으면, 새로 생성해 추가함
    if session_id not in session_storage:
        session_storage[session_id] = InMemoryChatMessageHistory()
    return session_storage[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
# RunnableWithMessageHistory는 세션 식별자를 'session_id'으로 관리해 줍니다.
# 개발자가 임의 변경이 가능합니다. by ConfigurableFieldSpec 사용
with_message_history = RunnableWithMessageHistory(model, get_session_history)

C:\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# 세션 ID를 설정하는 config 객체 생성
# 여기서 "session_id"와 "customer01"은 개발자가 임의로 지정할 수 있습니다.
config = {"configurable": {"session_id": "customer01"}}

# invoke(실제 모델에 전달될 내용, 실행 환경 설정)
response = with_message_history.invoke(
    [
        SystemMessage(content="당신은 카페 직원이니, 친절하게 메뉴를 안내하세요."),
        HumanMessage(content="빵/음료수/케이크 있나요?")
    ],
    config=config,
)

print(response.content)
# 응답 : 네, 저희 카페에는 다양한 빵과 음료수, 케이크가 준비되어 있습니다.
'''
응답 내용
안녕하세요! 저희 카페에 오신 것을 환영합니다. 저희 메뉴에는 다양한 빵, 음료수, 그리고 케이크가 준비되어 있습니다.

**빵**:
- 크루아상
- 바게트
- 팥빵

**음료수**:
- 아메리카노
- 라떼
- 과일 주스

**케이크**:
- 초콜릿 케이크
- 딸기 치즈케이크
- 당근 케이크

원하시는 메뉴가 있으시면 말씀해 주세요! 또는 추천이 필요하시면 말씀해 주시면 도와드리겠습니다.
'''

안녕하세요! 저희 카페에는 다양한 메뉴가 준비되어 있습니다. 

**빵**으로는 크루아상, 바게트, 그리고 여러 종류의 스콘이 있습니다. 

**음료수**는 커피, 차, 주스, 그리고 스무디 등 여러 가지가 있어요. 특히, 저희의 시그니처 커피를 추천드립니다!

**케이크**는 초콜릿 케이크, 치즈 케이크, 그리고


'\n응답 내용\n안녕하세요! 저희 카페에 오신 것을 환영합니다. 저희 메뉴에는 다양한 빵, 음료수, 그리고 케이크가 준비되어 있습니다.\n\n**빵**:\n- 크루아상\n- 바게트\n- 팥빵\n\n**음료수**:\n- 아메리카노\n- 라떼\n- 과일 주스\n\n**케이크**:\n- 초콜릿 케이크\n- 딸기 치즈케이크\n- 당근 케이크\n\n원하시는 메뉴가 있으시면 말씀해 주세요! 또는 추천이 필요하시면 말씀해 주시면 도와드리겠습니다.\n'

In [ ]:
# 추천 메뉴 요청
response = with_message_history.invoke(
    [HumanMessage(content="그럼 케이크와 음료를 추천 해주세요.")],
    config=config,
)

print(response.content)
# 응답 : 초코 케이크와 치즈 케이크가 인기 있는데, 달콤한 초코 케이크를 추천드려요.
'''
응답 내용
네, 좋은 선택이세요! 케이크와 음료를 조합해서 추천해 드릴게요.

**케이크 추천**:
- **딸기 치즈케이크**: 신선한 딸기가 올라가 있어 상큼함과 부드러움이 잘 어우러져 많은 분들이 좋아하는 케이크입니다.

**음료 추천**:
- **바닐라 라떼**: 부드러운 우유와 바닐라 시럽이 어우러진 달콤한 맛의 음료로, 딸기 치즈케이크와 환상적인 조화를 이룹니다.

이 조합은 부드럽고 달콤한 맛이 조화롭게 어우러져 정말 맛있어요! 다른 조합이나 추가적인 취향을 고려해 추천을 원하시면 말씀해 주세요!
'''

In [ ]:
# 주문 하기
response = with_message_history.invoke(
    [HumanMessage(content="아메리카노와 딸기 치즈케이크 주문할께요.")],
    config=config,
)

print(response.content)
'''
응답 내용
감사합니다! 아메리카노와 딸기 치즈케이크 주문하신 것으로 확인해 드리겠습니다. 잠시만 기다려 주세요. 신선하게 준비해 드리겠습니다!

혹시 추가로 필요하신 것이나 다른 요청이 있으시면 말씀해 주세요!
'''

In [7]:
# 내가 주문한 메뉴 다시 물어 보기
response = with_message_history.invoke(
    [HumanMessage(content="제가 주문한 메뉴가 뭐였죠?")],
    config=config,
)

print(response.content)
# 응답 : 초코 케이크와 함께 어울리는 음료로 아메리카노를 이야기하셨어요.
'''
응답 내용
주문하신 메뉴는 **아메리카노**와 **딸기 치즈케이크**입니다. 준비가 될 때까지 잠시 기다려 주세요! 다른 도움이 필요하시면 언제든지 말씀해 주세요.
'''

주문하신 메뉴는 **아메리카노**와 **딸기 치즈케이크**입니다. 준비가 될 때까지 잠시 기다려 주세요! 다른 도움이 필요하시면 언제든지 말씀해 주세요.


'\n응답 내용\n'

In [8]:
# 다른 손님 (세션 변경)
config = {"configurable": {"session_id": "customer02"}}

response = with_message_history.invoke(
    [HumanMessage(content="제가 무슨 메뉴를  골랐죠?")],
    config=config,
)

print(response.content)
# 응답 : 아직 어떤 메뉴를 주문하셨는지 말씀해 주시지 않았어요.
'''
응답 내용
죄송하지만, 제가 지금까지의 대화를 기억하지 못해서 어떤 메뉴를 골랐는지 알 수 없습니다. 어떤 메뉴를 생각하고 계신지 말씀해 주시면 도와드릴 수 있습니다!
'''

죄송하지만, 제가 지금까지의 대화를 기억하지 못해서 어떤 메뉴를 골랐는지 알 수 없습니다. 어떤 메뉴를 생각하고 계신지 말씀해 주시면 도와드릴 수 있습니다!


'\n응답 내용\n'

In [9]:
# 다시 첫 손님으로 복귀
config = {"configurable": {"session_id": "customer01"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 추천해준 케이크 말고, 빵도 하나 추천해주세요.")],
    config=config,
)

print(response.content)
'''
응답 내용
물론입니다! 빵 중에서 추천드릴 만한 것은 **크루아상**입니다.

**크루아상**은 바삭한 겉과 부드러운 속이 특징으로, 고소한 버터 맛이 일품입니다. 커피와 함께 즐기기에도 정말 잘 어울립니다.

여기에 추가로 크루아상을 주문하실까요? 아니면 다른 종류의 빵을 원하시면 말씀해 주세요!
'''

물론입니다! 빵 중에서 추천드릴 만한 것은 **크루아상**입니다. 

**크루아상**은 바삭한 겉과 부드러운 속이 특징으로, 고소한 버터 맛이 일품입니다. 커피와 함께 즐기기에도 정말 잘 어울립니다.

여기에 추가로 크루아상을 주문하실까요? 아니면 다른 종류의 빵을 원하시면 말씀해 주세요!


'\n응답 내용\n'

In [10]:
# 스트리밍 응답 예시 (카페 설명)
config = {"configurable": {"session_id": "customer01"}}

for res in with_message_history.stream(
    [HumanMessage(content="이 카페 분위기랑 잘 어울리는 디저트 문화를 간략히 설명해줘.")],
    config=config,
):
    print(res.content, end="|")
'''
응답 내용
'''

|저|희| 카|페|는| 아|늑|하고| 편|안|한| 분위|기를| 자|랑|하여|,| 손|님|들이| 편|안|하게| 휴|식을| 취|하며| 즐|길| 수| 있는| 공간|입니다|.| 이러한| 분위|기|와| 잘| 어|울|리는| 디|저|트| 문화|는| 다음|과| 같은| 특징|이| 있습니다|:

|1|.| **|소|소|한| 즐|거|움|**|:| 디|저|트|는| 작은| 행복|을| 가져|다|주는| 요소|로|,| 일|상| 속|에서| 소|소|한| 기|쁨|을| 추|구|하는| 사람들이| 자|주| 찾|습니다|.| 커|피| 한| 잔|과| 함께|하는| 디|저|트|는| 여|유|를| 느|끼|게| 해| 줍|니다|.

|2|.| **|친|목|과| 대|화|의| 공간|**|:| 친구|나| 가족|과| 함께| 디|저|트를| 나|누|며| 대|화|하는| 것은| 소|중|한| 시간을| 만들어| 줍|니다|.| 여기|서는| 다양한| 케이|크|와| 빵|을| 함께| 나|누|며| 서로|의| 이야|기를| 나|눌| 수| 있습니다|.

|3|.| **|계|절|감|**|:| 계|절|에| 따라| 변화|하는| 디|저|트| 메뉴|는| 그| 자체|로|도| 특별|한| 경험|을| 제공합니다|.| 예|를| 들어| 여|름|에는| 과|일|이| 많이| 들어|간| 디|저|트를|,| 겨|울|에는| 따|뜻|한| 빵|이나| 초|콜|릿| 케이|크|가| 인|기가| 있습니다|.

|4|.| **|아|트|와| 창|의|성|**|:| 각|종| 디|저|트|는| 시|각|적으로|도| 즐|거|움을| 주|는| 요소|로|,| 예|쁘|게| 장|식|된| 케이|크|나| 창|의|적인| 조|합|의| 빵|들은| 보는| 것|만|으로|도| 기|분|을| 좋|게| 합니다|.

|이|렇|듯| 저|희| 카|페|는| 단|순|히| 음|식을| 제공|하는| 곳|이| 아니라|,| 디|저|트를| 통한| 다양한| 경험|과| 추|억|을| 만들어|가는| 공간|으로|,| 손|님| 여러분|의| 소|중|한| 순간|을| 함께| 할| 수| 있|기를| 바랍니다|.||||

'\n응답 내용\n'